<a href="https://colab.research.google.com/github/Alok224/Celebal_Weekly_Assignments/blob/main/week2_alok_assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tesla EV Deliveries Analysis & Forecasting (2015–2025)

**Minor Project | Machine Learning | B.Tech CSE**

---

## Introduction

In this project, I am analyzing Tesla's EV delivery and production data from 2015 to 2025. The dataset contains information like deliveries, production units, average price, battery capacity, range, CO2 saved, and charging stations across different regions and models.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Load the dataset
df = pd.read_csv('/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv')
print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv'

## Basic Dataset Exploration

In [ ]:
print('Dataset Info')
df.info()
print('\n Basic Statistics')
df.describe()

In [ ]:
print('Null values in each column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## Data Cleaning


In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

In [ ]:
# Fill missing categorical values with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print('Cleaning done!')
print('Final shape:', df.shape)
print('Remaining nulls:', df.isnull().sum().sum())

## Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Deliveries trend over years
year_del = df.groupby('Year')['Estimated_Deliveries'].sum().reset_index()
axes[0, 0].plot(year_del['Year'], year_del['Estimated_Deliveries'], marker='o', color='steelblue')
axes[0, 0].set_title('Total Deliveries per Year')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Deliveries')

# 2. Production trend over years
year_prod = df.groupby('Year')['Production_Units'].sum().reset_index()
axes[0, 1].plot(year_prod['Year'], year_prod['Production_Units'], marker='s', color='orange')
axes[0, 1].set_title('Total Production per Year')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Production Units')

# 3. Average price trend
year_price = df.groupby('Year')['Avg_Price_USD'].mean().reset_index()
axes[1, 0].plot(year_price['Year'], year_price['Avg_Price_USD'], marker='^', color='green')
axes[1, 0].set_title('Average Price per Year (USD)')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Avg Price')

# 4. Deliveries by Model
model_del = df.groupby('Model')['Estimated_Deliveries'].sum().sort_values(ascending=False)
axes[1, 1].bar(model_del.index, model_del.values, color='coral')
axes[1, 1].set_title('Total Deliveries by Model')
axes[1, 1].set_xlabel('Model')
axes[1, 1].set_ylabel('Deliveries')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0])
axes[0].set_title('Correlation Heatmap')

# Deliveries by Region
region_del = df.groupby('Region')['Estimated_Deliveries'].sum().sort_values(ascending=False)
axes[1].bar(region_del.index, region_del.values, color='mediumpurple')
axes[1].set_title('Total Deliveries by Region')
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Deliveries')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
# 1. Delivery efficiency = deliveries / production
df['Delivery_Efficiency'] = df['Estimated_Deliveries'] / df['Production_Units']

# 2. CO2 saved per delivery
df['CO2_Per_Delivery'] = df['CO2_Saved_tons'] / df['Estimated_Deliveries']

# 3. Price per km of range
df['Price_Per_km'] = df['Avg_Price_USD'] / df['Range_km']

# 4. Encode categorical columns
le = LabelEncoder()
df['Region_enc'] = le.fit_transform(df['Region'])
df['Model_enc'] = le.fit_transform(df['Model'])
df['Source_enc'] = le.fit_transform(df['Source_Type'])

print('New features added:')
print(df[['Delivery_Efficiency', 'CO2_Per_Delivery', 'Price_Per_km']].describe())

## Machine Learning

In [ ]:
# Feature selection
features = ['Year', 'Month', 'Production_Units', 'Avg_Price_USD',
            'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons',
            'Charging_Stations', 'Region_enc', 'Model_enc',
            'Delivery_Efficiency', 'Price_Per_km']

X = df[features]
y = df['Estimated_Deliveries']

# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape, '| Test size:', X_test.shape)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('\nModels trained successfully!')

## Model Evaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name}')
    print(f'  MAE  : {mae:.2f}')
    print(f'  RMSE : {rmse:.2f}')
    print(f'  R²   : {r2:.4f}')
    print()

evaluate('Linear Regression', y_test, y_pred_lr)
evaluate('Random Forest', y_test, y_pred_rf)

## Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestRegressor(random_state=42),
                           param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV R²:', round(grid_search.best_score_, 4))

# Evaluate tuned model
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
evaluate('Tuned Random Forest', y_test, y_pred_best)

## Feature Importance

In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 3 features:', importances.index[:3].tolist())

## Time Series Forecasting using ARIMA

In [ ]:
# Aggregate total deliveries per year
ts_data = df.groupby('Year')['Estimated_Deliveries'].sum()

# Fit ARIMA model
arima_model = ARIMA(ts_data, order=(1, 1, 1))
arima_result = arima_model.fit()

# Forecast next 3 years
forecast = arima_result.forecast(steps=3)
forecast_years = [ts_data.index[-1] + i for i in range(1, 4)]


In [ ]:
# Plot
plt.figure(figsize=(10, 5))
plt.plot(ts_data.index, ts_data.values, marker='o', label='Actual', color='steelblue')
plt.plot(forecast_years, forecast.values, marker='s', linestyle='--', label='Forecast', color='red')
plt.title('ARIMA Forecast - Next 3 Years')
plt.xlabel('Year')
plt.ylabel('Total Deliveries')
plt.legend()
plt.tight_layout()
plt.show()

print('Forecasted Deliveries:')
for yr, val in zip(forecast_years, forecast.values):
    print(f'  {yr}: {int(val):,}')

## Conclusion

Here's a summary of what I found in this project:

**Best Model:** Random Forest (after GridSearchCV tuning) gave the best R² score compared to Linear Regression. Random Forest handled non-linear patterns in the data much better.

**Important Features:** The top features were `Production_Units`, `Delivery_Efficiency`, and `CO2_Saved_tons`. This makes sense because deliveries are closely tied to production capacity.

**Future Forecast (ARIMA):** The ARIMA model forecasts a continuing upward trend in Tesla's deliveries over the next 3 years, suggesting sustained growth.

**Key Observations from EDA:**
- Deliveries and production have grown steadily from 2015 to 2025
- North America is the largest region by deliveries
- Model 3 and Model Y have the highest delivery volumes
- Average prices show some fluctuation, possibly due to Tesla's pricing strategy changes

This project can be further improved by using more advanced models like XGBoost or LSTM for time series forecasting.